In [1]:
import sys
sys.path.insert(0, "/home/ci2dt2-ai/Proyectos/Psiquiatria")
import kd_common as kdc
import pandas as pd
import numpy as np
import pickle
import xgboost, sklearn, torch

cfg = kdc.get_config("ansiedad")
kdc.set_global_seed(kdc.SEED)

master = pd.read_csv(cfg.experiment_dir / "data" / "master_table.csv", dtype={"patient_id": str, "audio_id": str})
split = pd.read_csv(cfg.experiment_dir / "data" / "development_test_split.csv", dtype={"patient_id": str})
folds = pd.read_csv(cfg.experiment_dir / "data" / "folds_5cv.csv", dtype={"patient_id": str})
dev = kdc.build_dev_frame(master, split, folds)
test = kdc.build_test_frame(master, split)
oof = pd.read_csv(cfg.experiment_dir / "teacher" / "teacher_predictions_development_oof.csv", dtype={"patient_id": str})

selections = {}
for fs in ["librosa", "egemaps"]:
    selections[fs] = kdc.load_json(cfg.experiment_dir / "features" / fs / "selection.json")
print(selections)

{'librosa': {'feature_set': 'librosa', 'base_winner': 'DecisionTreeClassifier', 'kd_winner': 'ElasticNet', 'best_alpha': 0.0, 'base_threshold': 0.2, 'kd_threshold': 0.2, 'base_roc_auc_mean': 0.5011554029888924, 'kd_roc_auc_mean_at_alpha0.5': 0.5}, 'egemaps': {'feature_set': 'egemaps', 'base_winner': 'AdaBoostClassifier', 'kd_winner': 'AdaBoostRegressor', 'best_alpha': 0.0, 'base_threshold': 0.33, 'kd_threshold': 0.23, 'base_roc_auc_mean': 0.4986425726217195, 'kd_roc_auc_mean_at_alpha0.5': 0.5416398826323867}}


## Fase 13 / Paso 16 — Teacher final (todo Desarrollo) y predicción en Test

In [2]:
teacher_models, teacher_test_patient = kdc.train_patient_teacher_final_ensemble_and_predict_test(
    "ansiedad", dev, test,
)
teacher_predictions_test = kdc.broadcast_patient_scores(
    test[["sample_id", "patient_id", "label"]].rename(columns={"label": "y_real"}),
    teacher_test_patient, "p_teacher_test",
)
teacher_predictions_test.to_csv(cfg.experiment_dir / "teacher" / "teacher_predictions_test.csv", index=False)
teacher_test_patient.to_csv(cfg.experiment_dir / "teacher" / "teacher_test_patient.csv", index=False)
from sklearn.metrics import roc_auc_score
print(f"Teacher Test AUC (paciente, n={len(teacher_test_patient)}): {roc_auc_score(teacher_test_patient.y_real, teacher_test_patient.p_teacher_test):.3f}")

20:24:19 [INFO]    Cargados: 79 pacientes | shape=(79, 157, 1024) | clase0=60 | clase1=19


Teacher Test AUC (paciente, n=16): 0.312


## Fases 12-14 — Entrenar Base y KD finales por feature_set y predecir Test

In [3]:
predictions_all = {}

for fs in ["librosa", "egemaps"]:
    sel = selections[fs]
    out_dir = cfg.experiment_dir / "features" / fs

    with open(out_dir / "base_template_pipe.pkl", "rb") as f:
        base_template = pickle.load(f)
    with open(out_dir / "kd_template_pipe.pkl", "rb") as f:
        kd_template = pickle.load(f)
    with open(out_dir / "dev_feature_medians.pkl", "rb") as f:
        medians = pickle.load(f)

    X_dev, feat_cols, _ = kdc.load_feature_matrix(cfg, fs, dev, medians=medians)
    X_test, _, _ = kdc.load_feature_matrix(cfg, fs, test, medians=medians)
    y_dev = dev["label"].to_numpy().astype(float)

    p_teacher_oof = dev[["sample_id"]].merge(oof[["sample_id", "p_teacher_OOF"]], on="sample_id", how="left")["p_teacher_OOF"].to_numpy()
    y_kd_dev = kdc.build_kd_targets(y_dev, p_teacher_oof, sel["best_alpha"])

    base_final = kdc.fit_lazy_final(base_template, X_dev, y_dev.astype(int))
    kd_final = kdc.fit_lazy_final(kd_template, X_dev, y_kd_dev)

    p_base_test = kdc.predict_lazy(base_final, X_test)
    p_kd_test = kdc.predict_lazy(kd_final, X_test)

    preds = test[["sample_id", "patient_id", "label"]].rename(columns={"label": "y_real"}).copy()
    preds["p_base_test"] = p_base_test
    preds["p_kd_test"] = p_kd_test
    preds.to_csv(out_dir / "predictions_test.csv", index=False)
    predictions_all[fs] = preds

    with open(out_dir / "base_final_model.pkl", "wb") as f:
        pickle.dump(base_final, f)
    with open(out_dir / "kd_final_model.pkl", "wb") as f:
        pickle.dump(kd_final, f)

    from sklearn.metrics import roc_auc_score
    print(f"{fs}: Base test AUC={roc_auc_score(preds.y_real, preds.p_base_test):.3f} | KD test AUC={roc_auc_score(preds.y_real, preds.p_kd_test):.3f}")

librosa: Base test AUC=0.527 | KD test AUC=0.500


egemaps: Base test AUC=0.651 | KD test AUC=0.470


## Fase 15 — Tabla final de métricas en Test

In [4]:
rows = []
rows.append({"model": "BiLSTM_Teacher", **kdc.compute_binary_metrics(
    teacher_predictions_test["y_real"].to_numpy(), teacher_predictions_test["p_teacher_test"].to_numpy(), threshold=0.5)})

for fs in ["librosa", "egemaps"]:
    sel = selections[fs]
    preds = predictions_all[fs]
    rows.append({"model": f"XGB_Base_{fs}", **kdc.compute_binary_metrics(
        preds["y_real"].to_numpy(), preds["p_base_test"].to_numpy(), threshold=sel["base_threshold"])})
    rows.append({"model": f"XGB_KD_{fs}", **kdc.compute_binary_metrics(
        preds["y_real"].to_numpy(), preds["p_kd_test"].to_numpy(), threshold=sel["kd_threshold"])})

test_metrics = pd.DataFrame(rows)
test_metrics.to_csv(cfg.experiment_dir / "results" / "test_metrics.csv", index=False)
test_metrics

,model,roc_auc,pr_auc,accuracy,precision,recall,specificity,f1,brier
0,BiLSTM_Teacher,0.284022,0.180266,0.794118,0.000000,0.000000,1.000000,0.000000,0.196671
1,XGB_Base_librosa,0.526911,0.216114,0.657201,0.239382,0.305419,0.748404,0.268398,0.342799
2,XGB_KD_librosa,0.500000,0.205882,0.205882,0.205882,1.000000,0.000000,0.341463,0.164606
3,XGB_Base_egemaps,0.651303,0.307725,0.278905,0.220879,0.990148,0.094508,0.361186,0.205975
4,XGB_KD_egemaps,0.470437,0.184961,0.333671,0.198939,0.738916,0.228608,0.313480,0.170875


## Fase 18 — Tabla resumen de validación cruzada (Desarrollo)

In [5]:
teacher_cv = pd.read_csv(cfg.experiment_dir / "teacher" / "teacher_metrics_cv.csv")
rows = [{
    "model": "BiLSTM", "feature_set": "-", "alpha": "-",
    "roc_auc_mean": teacher_cv["roc_auc"].mean(), "roc_auc_std": teacher_cv["roc_auc"].std(),
    "f1_mean": teacher_cv["f1"].mean(), "f1_std": teacher_cv["f1"].std(),
}]

for fs in ["librosa", "egemaps"]:
    sel = selections[fs]
    alpha_df = pd.read_csv(cfg.experiment_dir / "features" / fs / "alpha_results_cv.csv")
    rows.append({
        "model": f"Student_Base ({sel['base_winner']})", "feature_set": fs, "alpha": 1.0,
        "roc_auc_mean": sel["base_roc_auc_mean"], "roc_auc_std": np.nan,
        "f1_mean": np.nan, "f1_std": np.nan,
    })
    for _, r in alpha_df.iterrows():
        label = "Student_Soft" if r["alpha"] == 0.0 else "Student_KD"
        rows.append({
            "model": f"{label} ({sel['kd_winner']})", "feature_set": fs, "alpha": r["alpha"],
            "roc_auc_mean": r["roc_auc_mean"], "roc_auc_std": r["roc_auc_std"],
            "f1_mean": r["f1_mean"], "f1_std": r["f1_std"],
        })

cv_table = pd.DataFrame(rows)
cv_table.to_csv(cfg.experiment_dir / "results" / "cross_validation_metrics.csv", index=False)
cv_table

,model,feature_set,alpha,roc_auc_mean,roc_auc_std,f1_mean,f1_std
0,BiLSTM,-,-,0.588148,0.140385,0.000000,0.000000
1,Student_Base (DecisionTreeClassifier),librosa,1.0,0.501155,NaN,NaN,NaN
2,Student_Soft (ElasticNet),librosa,0.0,0.500000,0.000000,0.000000,0.000000
3,Student_KD (ElasticNet),librosa,0.25,0.500000,0.000000,0.000000,0.000000
4,Student_KD (ElasticNet),librosa,0.5,0.500000,0.000000,0.000000,0.000000
5,Student_KD (ElasticNet),librosa,0.75,0.500000,0.000000,0.000000,0.000000
6,Student_KD (ElasticNet),librosa,1.0,0.500000,0.000000,0.000000,0.000000
7,Student_Base (AdaBoostClassifier),egemaps,1.0,0.498643,NaN,NaN,NaN
8,Student_Soft (AdaBoostRegressor),egemaps,0.0,0.647073,0.173338,0.000000,0.000000
9,Student_KD (AdaBoostRegressor),egemaps,0.25,0.566305,0.147677,0.000000,0.000000


## Reproducibilidad: congelar configuración

In [6]:
config = {
    "disorder": "ansiedad",
    "seed": kdc.SEED,
    "n_dev_patients": int(dev.patient_id.nunique()),
    "n_test_patients": int(test.patient_id.nunique()),
    "teacher_config": kdc.WINNING_BILSTM_CONFIG,
    "teacher_oof_n_repeats": 10,
    "curated_classifiers": [c.__name__ for c in kdc.CURATED_CLASSIFIERS],
    "curated_regressors": [c.__name__ for c in kdc.CURATED_REGRESSORS],
    "selections": selections,
    "library_versions": {
        "xgboost": xgboost.__version__, "sklearn": sklearn.__version__, "torch": torch.__version__,
    },
}
kdc.save_json(config, cfg.experiment_dir / "results" / "experiment_config.json")
config

{'disorder': 'ansiedad',
 'seed': 42,
 'n_dev_patients': 63,
 'n_test_patients': 16,
 'teacher_config': {'emb_model': 'wav2vec2-large-robust',
  'hidden': 128,
  'pooling': 'meanmax',
  'loss_type': 'bce_posw',
  'use_warmup': False,
  'dropout_p': 0.4,
  'mixup': 'fix',
  'pca': False},
 'teacher_oof_n_repeats': 10,
 'curated_classifiers': ['LogisticRegression',
  'RandomForestClassifier',
  'ExtraTreesClassifier',
  'GradientBoostingClassifier',
  'AdaBoostClassifier',
  'BaggingClassifier',
  'DecisionTreeClassifier',
  'KNeighborsClassifier',
  'GaussianNB',
  'QuadraticDiscriminantAnalysis',
  'LinearDiscriminantAnalysis',
  'XGBClassifier'],
 'curated_regressors': ['LinearRegression',
  'Ridge',
  'Lasso',
  'ElasticNet',
  'RandomForestRegressor',
  'ExtraTreesRegressor',
  'GradientBoostingRegressor',
  'AdaBoostRegressor',
  'BaggingRegressor',
  'DecisionTreeRegressor',
  'KNeighborsRegressor',
  'XGBRegressor'],
 'selections': {'librosa': {'feature_set': 'librosa',
   'base_